# Dataset Construction

This notebook builds the combined SF crime dataset (2003–present) that serves as the foundation for Assignment 1.

**Steps:**
1. Load and inspect the 2018-present dataset
2. Clean and keep only relevant columns
3. Load and inspect the historical 2003–2018 dataset
4. Map categories across the two schemas
5. Merge, deduplicate, and save

In [1]:
import pandas as pd

df_recent = pd.read_csv('Police_Department_Incident_Reports__2018_to_Present.csv')

print(df_recent.shape)
df_recent.head(3)

(1011597, 29)


,Row ID,Incident Datetime,Incident Date,Incident Time,Incident Year,Incident Day of Week,Report Datetime,Incident ID,Incident Number,CAD Number,...,CNN,Police District,Analysis Neighborhood,Supervisor District,Supervisor District 2012,Latitude,Longitude,Point,data_as_of,data_loaded_at
0,152278216710,2025/10/31 05:55:00 PM,2025/10/31,17:55,2025,Friday,2025/10/31 05:55:00 PM,1522782,250613625,253042598.0,...,24074000.0,Mission,Mission,9.0,9.0,37.752270,-122.417877,POINT (-122.417877197 37.752269745),2025/11/04 09:37:41 AM,2025/11/05 09:59:28 AM
1,148998204134,2025/06/13 12:41:00 PM,2025/06/13,12:41,2025,Friday,2025/06/13 12:46:00 PM,1489982,250329888,251641497.0,...,20872000.0,Ingleside,McLaren Park,9.0,9.0,37.718128,-122.414177,POINT (-122.414176941 37.718128204),2025/06/14 09:37:39 AM,2025/06/15 09:53:25 AM
2,152328663010,2025/11/03 02:00:00 PM,2025/11/03,14:00,2025,Monday,2025/11/03 02:00:00 PM,1523286,250269557,NaN,...,NaN,Out of SF,NaN,NaN,NaN,NaN,NaN,NaN,2025/11/04 09:37:41 AM,2025/11/05 09:59:28 AM


In [2]:
# Re-parse with explicit format (faster + no warning)
df_recent['Incident Datetime'] = pd.to_datetime(
    df_recent['Incident Datetime'], format='%Y/%m/%d %I:%M:%S %p'
)

# Drop incomplete year 2026
df_recent = df_recent[df_recent['Incident Datetime'].dt.year <= 2025]

print(f"Rows after dropping 2026: {len(df_recent):,}")
print(f"Date range: {df_recent['Incident Datetime'].min().date()} → {df_recent['Incident Datetime'].max().date()}")

Rows after dropping 2026: 998,893
Date range: 2018-01-01 → 2025-12-31


In [3]:
cols_to_keep = [
    'Incident Datetime',
    'Incident Category',
    'Police District',
    'Latitude',
    'Longitude',
    'Incident Description'
]

df_recent = df_recent[cols_to_keep]
print(df_recent.shape)
df_recent.head(3)

(998893, 6)


,Incident Datetime,Incident Category,Police District,Latitude,Longitude,Incident Description
0,2025-10-31 17:55:00,Drug Offense,Mission,37.752270,-122.417877,"Narcotics Paraphernalia, Possession of"
1,2025-06-13 12:41:00,Assault,Ingleside,37.718128,-122.414177,Battery
2,2025-11-03 14:00:00,Warrant,Out of SF,NaN,NaN,"Warrant Arrest, Local SF Warrant"


In [4]:
# Drop rows where Incident Category is missing
num_dropped = len(df_recent) - len(df_recent.dropna(subset=['Incident Category']))
# Print the number of rows dropped
print(f"Number of rows dropped due to missing Incident Category: {num_dropped}")

df_recent = df_recent.dropna(subset=['Incident Category'])

# Show all categories and their counts
category_counts = df_recent['Incident Category'].value_counts()
print(f"Number of unique categories: {len(category_counts)}\n")
print(category_counts.to_string())

Number of rows dropped due to missing Incident Category: 1477
Number of unique categories: 49

Incident Category
Larceny Theft                                   294099
Other Miscellaneous                              68985
Malicious Mischief                               68455
Assault                                          64569
Burglary                                         55917
Motor Vehicle Theft                              54814
Recovered Vehicle                                40244
Non-Criminal                                     37933
Fraud                                            33866
Warrant                                          32987
Lost Property                                    30890
Drug Offense                                     30607
Missing Person                                   22542
Robbery                                          22489
Suspicious Occ                                   21300
Disorderly Conduct                               18589
Offence

# Loading the old dataset

In [5]:
df_hist = pd.read_csv('Police_Department_Incident_Reports__Historical_2003_to_May_2018.csv')

print(df_hist.shape)
print("\nColumns:", df_hist.columns.tolist())
df_hist.head(3)

(2071736, 15)

Columns: ['PdId', 'IncidntNum', 'Incident Code', 'Category', 'Descript', 'DayOfWeek', 'Date', 'Time', 'PdDistrict', 'Resolution', 'Address', 'X', 'Y', 'location', 'data_loaded_at']


,PdId,IncidntNum,Incident Code,Category,Descript,DayOfWeek,Date,Time,PdDistrict,Resolution,Address,X,Y,location,data_loaded_at
0,11049313327195,110493133,27195,TRESPASS,TRESPASSING,Sunday,06/19/2011,13:06,TARAVAL,NONE,100 Block of APTOS AV,-122.466758,37.729185,POINT (-122.466758005 37.72918458),2025/06/20 12:17:56 PM
1,16020415607021,160204156,7021,VEHICLE THEFT,STOLEN AUTOMOBILE,Thursday,03/03/2016,19:30,TARAVAL,NONE,100 Block of BEPLER ST,-122.463545,37.707968,POINT (-122.463545017 37.707968365),2025/06/20 12:17:56 PM
2,6102672004134,61026720,4134,ASSAULT,BATTERY,Monday,09/25/2006,22:15,NORTHERN,NONE,400 Block of FULTON ST,-122.425839,37.778486,POINT (-122.42583948 37.778486375),2025/06/20 12:17:56 PM


In [6]:
# Combine separate Date and Time columns into a single datetime
df_hist['Incident Datetime'] = pd.to_datetime(
    df_hist['Date'] + ' ' + df_hist['Time'], format='%m/%d/%Y %H:%M'
)

# Check the date range and which years are present
print("Earliest date:", df_hist['Incident Datetime'].min())
print("Latest date:  ", df_hist['Incident Datetime'].max())
print("Years present:", sorted(df_hist['Incident Datetime'].dt.year.unique()))

# Drop 2018 - it only runs to May, the recent dataset covers 2018 fully
df_hist = df_hist[df_hist['Incident Datetime'].dt.year < 2018]

print(f"\nRows after dropping 2018: {len(df_hist):,}")
print(f"Date range: {df_hist['Incident Datetime'].min().date()} → {df_hist['Incident Datetime'].max().date()}")

Earliest date: 2003-01-01 00:01:00
Latest date:   2018-05-15 10:30:00
Years present: [np.int32(2003), np.int32(2004), np.int32(2005), np.int32(2006), np.int32(2007), np.int32(2008), np.int32(2009), np.int32(2010), np.int32(2011), np.int32(2012), np.int32(2013), np.int32(2014), np.int32(2015), np.int32(2016), np.int32(2017), np.int32(2018)]

Rows after dropping 2018: 2,028,003
Date range: 2003-01-01 → 2017-12-31


In [7]:
# Keep only relevant columns and rename to match the recent dataset's naming
df_hist = df_hist[['Incident Datetime', 'Category', 'Descript', 'PdDistrict', 'X', 'Y']].rename(columns={
    'Category':   'Incident Category',
    'Descript':   'Incident Description',
    'PdDistrict': 'Police District',
    'X':          'Longitude',
    'Y':          'Latitude'
})

# Drop rows where Incident Category is missing
df_hist = df_hist.dropna(subset=['Incident Category'])

print(df_hist.shape)
df_hist.head(3)

(2028003, 6)


,Incident Datetime,Incident Category,Incident Description,Police District,Longitude,Latitude
0,2011-06-19 13:06:00,TRESPASS,TRESPASSING,TARAVAL,-122.466758,37.729185
1,2016-03-03 19:30:00,VEHICLE THEFT,STOLEN AUTOMOBILE,TARAVAL,-122.463545,37.707968
2,2006-09-25 22:15:00,ASSAULT,BATTERY,NORTHERN,-122.425839,37.778486


In [8]:
# Compare categories across both datasets
hist_cats = sorted(df_hist['Incident Category'].dropna().unique())
recent_cats = sorted(df_recent['Incident Category'].dropna().unique())

print(f"Historical categories ({len(hist_cats)}):")
for c in hist_cats:
    print(" ", c)

print(f"\nRecent categories ({len(recent_cats)}):")
for c in recent_cats:
    print(" ", c)

Historical categories (37):
  ARSON
  ASSAULT
  BAD CHECKS
  BRIBERY
  BURGLARY
  DISORDERLY CONDUCT
  DRIVING UNDER THE INFLUENCE
  DRUG/NARCOTIC
  DRUNKENNESS
  EMBEZZLEMENT
  EXTORTION
  FORGERY/COUNTERFEITING
  FRAUD
  GAMBLING
  KIDNAPPING
  LARCENY/THEFT
  LIQUOR LAWS
  LOITERING
  MISSING PERSON
  NON-CRIMINAL
  OTHER OFFENSES
  PORNOGRAPHY/OBSCENE MAT
  PROSTITUTION
  RECOVERED VEHICLE
  ROBBERY
  SECONDARY CODES
  SEX OFFENSES, FORCIBLE
  SEX OFFENSES, NON FORCIBLE
  STOLEN PROPERTY
  SUICIDE
  SUSPICIOUS OCC
  TREA
  TRESPASS
  VANDALISM
  VEHICLE THEFT
  WARRANTS
  WEAPON LAWS

Recent categories (49):
  Arson
  Assault
  Burglary
  Case Closure
  Civil Sidewalks
  Courtesy Report
  Disorderly Conduct
  Drug Offense
  Drug Violation
  Embezzlement
  Fire Report
  Forgery And Counterfeiting
  Fraud
  Gambling
  Homicide
  Human Trafficking (A), Commercial Sex Acts
  Human Trafficking (B), Involuntary Servitude
  Human Trafficking, Commercial Sex Acts
  Larceny Theft
  Liquor L

In [9]:
# Mapping for historical dataset: rename ALL CAPS old names to canonical modern names
hist_mapping = {
    'ARSON':                      'Arson',
    'ASSAULT':                    'Assault',
    'BURGLARY':                   'Burglary',
    'DISORDERLY CONDUCT':         'Disorderly Conduct',
    'DRIVING UNDER THE INFLUENCE':'Traffic Violation Arrest',
    'DRUG/NARCOTIC':              'Drug Offense',
    'EMBEZZLEMENT':               'Embezzlement',
    'FORGERY/COUNTERFEITING':     'Forgery And Counterfeiting',
    'FRAUD':                      'Fraud',
    'GAMBLING':                   'Gambling',
    'LARCENY/THEFT':              'Larceny Theft',
    'LIQUOR LAWS':                'Liquor Laws',
    'LOITERING':                  'Civil Sidewalks',
    'MISSING PERSON':             'Missing Person',
    'NON-CRIMINAL':               'Non-Criminal',
    'OTHER OFFENSES':             'Other Offenses',
    'PROSTITUTION':               'Prostitution',
    'RECOVERED VEHICLE':          'Recovered Vehicle',
    'ROBBERY':                    'Robbery',
    'SEX OFFENSES, FORCIBLE':     'Sex Offense',
    'SEX OFFENSES, NON FORCIBLE': 'Sex Offense',
    'STOLEN PROPERTY':            'Stolen Property',
    'SUICIDE':                    'Suicide',
    'SUSPICIOUS OCC':             'Suspicious Occ',
    'VANDALISM':                  'Vandalism',
    'VEHICLE THEFT':              'Motor Vehicle Theft',
    'WARRANTS':                   'Warrant',
    'WEAPON LAWS':                'Weapons Offense',
}

# Mapping for recent dataset: merge variant/duplicate names into one canonical name
recent_mapping = {
    'Drug Violation':                               'Drug Offense',
    'Weapons Offence':                              'Weapons Offense',
    'Weapons Carrying Etc':                         'Weapons Offense',
    'Motor Vehicle Theft?':                         'Motor Vehicle Theft',
    'Malicious Mischief':                           'Vandalism',
    'Rape':                                         'Sex Offense',
    'Suspicious':                                   'Suspicious Occ',
    'Human Trafficking (A), Commercial Sex Acts':   'Human Trafficking, Commercial Sex Acts',
    'Human Trafficking (B), Involuntary Servitude': 'Human Trafficking, Commercial Sex Acts',
}

# Apply mappings
df_hist['Incident Category'] = df_hist['Incident Category'].replace(hist_mapping)
df_recent['Incident Category'] = df_recent['Incident Category'].replace(recent_mapping)

# Sanity check - show historical categories that were NOT in the mapping (kept original name)
mapped = set(hist_mapping.keys())
all_hist_orig = set(hist_cats)  # from before mapping
unmapped = all_hist_orig - mapped
print("Historical categories left unmapped (kept original name):")
for c in sorted(unmapped):
    print(" ", c)

Historical categories left unmapped (kept original name):
  BAD CHECKS
  BRIBERY
  DRUNKENNESS
  EXTORTION
  KIDNAPPING
  PORNOGRAPHY/OBSCENE MAT
  SECONDARY CODES
  TREA
  TRESPASS


In [10]:
# Merge the two datasets
df = pd.concat([df_hist, df_recent], ignore_index=True)

print(f"Total rows: {len(df):,}")
print(f"Date range: {df['Incident Datetime'].min().date()} → {df['Incident Datetime'].max().date()}")
print(f"Unique categories: {df['Incident Category'].nunique()}")
df.head(3)

Total rows: 3,025,419
Date range: 2003-01-01 → 2025-12-31
Unique categories: 49


,Incident Datetime,Incident Category,Incident Description,Police District,Longitude,Latitude
0,2011-06-19 13:06:00,TRESPASS,TRESPASSING,TARAVAL,-122.466758,37.729185
1,2016-03-03 19:30:00,Motor Vehicle Theft,STOLEN AUTOMOBILE,TARAVAL,-122.463545,37.707968
2,2006-09-25 22:15:00,Assault,BATTERY,NORTHERN,-122.425839,37.778486


In [11]:
# Report and remove duplicates and rows with missing coordinates
n_dupes = df.duplicated().sum()
missing_coords = df[['Latitude', 'Longitude']].isna().any(axis=1).sum()
print(f"Duplicate rows:          {n_dupes:,}")
print(f"Missing coordinates:     {missing_coords:,}")

df = df.drop_duplicates()
df = df.dropna(subset=['Latitude', 'Longitude'])

# Normalise Police District capitalisation (historical dataset uses ALL CAPS)
df['Police District'] = df['Police District'].str.title()

print(f"\nRows after cleaning:     {len(df):,}")
print(f"Date range: {df['Incident Datetime'].min().date()} → {df['Incident Datetime'].max().date()}")
print(f"Unique categories: {df['Incident Category'].nunique()}")
print(f"Unique districts:  {sorted(df['Police District'].dropna().unique())}")

Duplicate rows:          37,056
Missing coordinates:     55,378

Rows after cleaning:     2,934,658
Date range: 2003-01-01 → 2025-12-31
Unique categories: 49
Unique districts:  ['Bayview', 'Central', 'Ingleside', 'Mission', 'Northern', 'Out Of Sf', 'Park', 'Richmond', 'Southern', 'Taraval', 'Tenderloin']


In [12]:
# Coerce all columns to plain numpy/Python types before saving.
# This avoids ArrowKeyError from pandas extension types (StringDtype, etc.)
df_out = df.copy()
df_out['Incident Datetime'] = df_out['Incident Datetime'].astype('datetime64[ns]')
df_out['Latitude']  = df_out['Latitude'].astype('float64')
df_out['Longitude'] = df_out['Longitude'].astype('float64')
for col in ['Incident Category', 'Incident Description', 'Police District']:
    df_out[col] = df_out[col].astype(object)  # plain Python str, NaN stays as float NaN

df_out.to_parquet('sf_crime_2003_2025.parquet', index=False)
print(f"Saved sf_crime_2003_2025.parquet  ({len(df_out):,} rows)")

Saved sf_crime_2003_2025.parquet  (2,934,658 rows)
